# Goal 3 — Consultas, atualizações e transações em MySQL

## Consultas em MySQL
Consultas com diferentes níveis de complexidade.

In [205]:
# %pip install mysql-connector-python pandas

from concurrent.futures import ThreadPoolExecutor
from threading import Barrier
from time import perf_counter
import pandas as pd

In [206]:
import mysql.connector
import pandas as pd
from getpass import getpass

connection = mysql.connector.connect(host="localhost", port=3306, user="root", password=getpass("Password MySQL: "), database="bda_movielens")

def consultar(sql):
    with connection.cursor() as cursor:
        cursor.execute(sql)
        return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)

## Q1 — Consulta simples
Filtragem e projeção. Consultar os filmes lançados após 2010.

In [207]:
q1 = """
SELECT movieId, title, year
FROM movies
WHERE year > 2010
ORDER BY movieId ASC;
"""

resultado_q1 = consultar(q1) 

print("Número de resultados:", len(q1))
display(q1)

Número de resultados: 81


'\nSELECT movieId, title, year\nFROM movies\nWHERE year > 2010\nORDER BY movieId ASC;\n'

## Q2 — Consulta básica
Filtragem, projeção e ordenação. Consultar os 20 filmes mais recentes.

In [208]:
q2 = """
SELECT movieId, title, year
FROM movies
WHERE year IS NOT NULL
ORDER BY year DESC, movieId ASC
LIMIT 20;
"""
resultado_q2 = consultar(q2)

print("Número de resultados:", len(q2))
display(q2)

Número de resultados: 106


'\nSELECT movieId, title, year\nFROM movies\nWHERE year IS NOT NULL\nORDER BY year DESC, movieId ASC\nLIMIT 20;\n'

## Q3 — Consulta intermédia
Junção, agregação e agrupamento. Calcular a classificação média e o número de avaliações de cada filme avaliado.

In [209]:
q3 ="""
SELECT m.movieId, m.title,
    ROUND(AVG(r.rating), 4) AS averageRating,
    COUNT(*) AS totalRatings
FROM movies AS m
INNER JOIN ratings AS r ON m.movieId = r.movieId
GROUP BY m.movieId, m.title
ORDER BY m.movieId ASC;
"""
resultado_q3 = consultar(q3)

print("Número de resultados:", len(q3))
display(q3)

Número de resultados: 221


'\nSELECT m.movieId, m.title,\n    ROUND(AVG(r.rating), 4) AS averageRating,\n    COUNT(*) AS totalRatings\nFROM movies AS m\nINNER JOIN ratings AS r ON m.movieId = r.movieId\nGROUP BY m.movieId, m.title\nORDER BY m.movieId ASC;\n'

## Q4 — Consulta avançada
Múltiplas junções, filtragem, agregação, agrupamento e ordenação. Consultar os 10 filmes de comédia com melhor classificação média e pelo menos 100 avaliações.

In [210]:
q4 ="""
SELECT m.movieId, m.title,
    ROUND(AVG(r.rating), 4) AS averageRating,
    COUNT(*) AS totalRatings
FROM movies AS m
INNER JOIN ratings AS r ON m.movieId = r.movieId
INNER JOIN movie_genres AS mg ON m.movieId = mg.movieId
INNER JOIN genres AS g ON mg.genreId = g.genreId
WHERE g.genre = 'Comedy'
GROUP BY m.movieId, m.title
HAVING COUNT(*) >= 100
ORDER BY averageRating DESC, m.movieId ASC
LIMIT 10;
"""
resultado_q4 = consultar(q4)

print("Número de resultados:", len(q4))
display(q4)

Número de resultados: 403


"\nSELECT m.movieId, m.title,\n    ROUND(AVG(r.rating), 4) AS averageRating,\n    COUNT(*) AS totalRatings\nFROM movies AS m\nINNER JOIN ratings AS r ON m.movieId = r.movieId\nINNER JOIN movie_genres AS mg ON m.movieId = mg.movieId\nINNER JOIN genres AS g ON mg.genreId = g.genreId\nWHERE g.genre = 'Comedy'\nGROUP BY m.movieId, m.title\nHAVING COUNT(*) >= 100\nORDER BY averageRating DESC, m.movieId ASC\nLIMIT 10;\n"

# Operações CRUD
Inserção, atualização e eliminação de dados de teste.

In [211]:
testUserId = 999999
testMovieId = 999999

def executar(sql, params=()):
    try:
        with connection.cursor() as cursor:
            cursor.execute(sql, params)
        connection.commit()
    except Exception:
        connection.rollback()
        raise

def verificar_ids(ids):
    for i in ids:
        if not consultar(f"SELECT userId FROM users WHERE userId = {i}").empty:
            raise ValueError(f"Utilizador {i} já existe.")
        if not consultar(f"SELECT movieId FROM movies WHERE movieId = {i} OR imdbId = {i} OR tmdbId = {i}").empty:
            raise ValueError(f"Identificador de filme {i} já existe.")
    connection.rollback()

verificar_ids([testUserId])

## I1 — Inserção de um utilizador

In [212]:
executar("INSERT INTO users (userId, country) VALUES (%s, %s)", (testUserId, "Portugal"))
display(consultar(f"SELECT * FROM users WHERE userId = {testUserId}"))

,userId,country
0,999999,Portugal


## I2 — Inserção de um filme

In [213]:
executar("INSERT INTO movies (movieId, title, year, imdbId, tmdbId) VALUES (%s, %s, %s, %s, %s)", (testMovieId, "Test Movie", 2026, testMovieId, testMovieId))
display(consultar(f"SELECT * FROM movies WHERE movieId = {testMovieId}"))

,movieId,title,year,imdbId,tmdbId
0,999999,Test Movie,2026,999999,999999


## I3 — Inserção de uma avaliação

In [214]:
executar("INSERT INTO ratings (userId, movieId, rating, timestamp) VALUES (%s, %s, %s, NOW())", (testUserId, testMovieId, 3.5))
display(consultar(f"SELECT * FROM ratings WHERE userId = {testUserId} AND movieId = {testMovieId}"))

,userId,movieId,rating,timestamp
0,999999,999999,3.5,2026-11-24 20:48:26


## U1 — Atualização do título

In [215]:
executar("UPDATE movies SET title = %s WHERE movieId = %s", ("Test Movie Updated", testMovieId))
display(consultar(f"SELECT * FROM movies WHERE movieId = {testMovieId}"))

,movieId,title,year,imdbId,tmdbId
0,999999,Test Movie Updated,2026,999999,999999


## U2 — Atualização da avaliação

In [216]:
executar("UPDATE ratings SET rating = %s WHERE userId = %s AND movieId = %s", (4.5, testUserId, testMovieId))
display(consultar(f"SELECT * FROM ratings WHERE userId = {testUserId} AND movieId = {testMovieId}"))

,userId,movieId,rating,timestamp
0,999999,999999,4.5,2026-11-24 20:48:26


## D1 — Eliminação da avaliação

In [217]:
executar("DELETE FROM ratings WHERE userId = %s AND movieId = %s", (testUserId, testMovieId))
display(consultar(f"SELECT * FROM ratings WHERE userId = {testUserId} AND movieId = {testMovieId}"))

,userId,movieId,rating,timestamp


## D2 — Eliminação do filme

In [218]:
executar("DELETE FROM movies WHERE movieId = %s", (testMovieId,))
display(consultar(f"SELECT * FROM movies WHERE movieId = {testMovieId}"))

,movieId,title,year,imdbId,tmdbId


## D3 — Eliminação do utilizador

In [219]:
executar("DELETE FROM users WHERE userId = %s", (testUserId,))
display(consultar(f"SELECT * FROM users WHERE userId = {testUserId}"))

,userId,country


# Transações
## T1 — Transação com sucesso (COMMIT)

Limpeza de dados

In [220]:
def limpar_dados_teste(i):

    connection.rollback()

    try:
        with connection.cursor() as cursor:

            # Eliminar avaliações
            cursor.execute("""
                DELETE FROM ratings
                WHERE userId = %s AND movieId = %s
            """, (i, i))

            # Eliminar filme
            cursor.execute("""
                DELETE FROM movies
                WHERE movieId = %s
            """, (i,))

            # Eliminar utilizador
            cursor.execute("""
                DELETE FROM users
                WHERE userId = %s
            """, (i,))

        connection.commit()

        print(f"Dados de teste {i} eliminados.")

    except mysql.connector.Error:
        connection.rollback()
        raise

### Verifica o estado das tabelas após as operações CRUD

In [221]:
def estado(i):

    users = consultar(
        f"SELECT * FROM users WHERE userId = {i}"
    )

    movies = consultar(
        f"SELECT * FROM movies WHERE movieId = {i}"
    )

    ratings = consultar(
        f"SELECT * FROM ratings WHERE movieId = {i}"
    )

    return {
        "users": len(users),
        "movies": len(movies),
        "ratings": len(ratings)
    }

### Testar transação com commit e rollback

Preparação da transação

In [222]:
i = 999991

limpar_dados_teste(i)
verificar_ids([i])

connection.rollback()
connection.start_transaction()

Dados de teste 999991 eliminados.


Execução da transação

In [223]:
try:
    with connection.cursor() as cursor:

        # Inserir um utilizador
        cursor.execute("""
            INSERT INTO users (userId, country)
            VALUES (%s, %s)
        """, (i, "Portugal"))

        # Inserir um filme
        cursor.execute("""
            INSERT INTO movies
            (movieId, title, year, imdbId, tmdbId)
            VALUES (%s, %s, %s, %s, %s)
        """, (i, "Transaction Test Movie", 2026, i, i))

        # Inserir uma avaliação
        cursor.execute("""
            INSERT INTO ratings
            (userId, movieId, rating, timestamp)
            VALUES (%s, %s, %s, NOW())
        """, (i, i, 4.5))

    connection.commit()
    print("COMMIT executado com sucesso.")

except mysql.connector.Error as erro:
    connection.rollback()
    print("Erro:", erro)
    raise

COMMIT executado com sucesso.


Verificação dos resultados

In [224]:
resultado = estado(i)

print(resultado)

assert resultado == {
    "users": 1,
    "movies": 1,
    "ratings": 1
}

connection.rollback()

{'users': 1, 'movies': 1, 'ratings': 1}


In [225]:
print(estado(999992))
connection.rollback()

{'users': 0, 'movies': 0, 'ratings': 0}


## T2 — Transação com falha e rollback automático

Preparação da transação

In [226]:
i = 999992

limpar_dados_teste(i)

verificar_ids([i])

connection.rollback()
connection.start_transaction()

Dados de teste 999992 eliminados.


Transação com falha (ROLLBACK)

In [227]:
try:
    with connection.cursor() as cursor:

        # Inserir um utilizador
        cursor.execute("""
            INSERT INTO users (userId, country)
            VALUES (%s, %s)
        """, (i, "Portugal"))

        # Inserir um filme
        cursor.execute("""
            INSERT INTO movies
            (movieId, title, year, imdbId, tmdbId)
            VALUES (%s, %s, %s, %s, %s)
        """, (i, "Rollback Test Movie", 2026, i, i))

        # Inserir uma avaliação válida
        cursor.execute("""
            INSERT INTO ratings
            (userId, movieId, rating, timestamp)
            VALUES (%s, %s, %s, NOW())
        """, (i, i, 3.5))

        # Provocar um erro de chave duplicada
        cursor.execute("""
            INSERT INTO ratings
            (userId, movieId, rating, timestamp)
            VALUES (%s, %s, %s, NOW())
        """, (i, i, 4.5))

except mysql.connector.Error as erro:

    connection.rollback()

    print("ROLLBACK executado.")
    print("Código do erro:", erro.errno)
    print("Mensagem:", erro.msg)

    if erro.errno != 1062:
        raise

else:
    connection.rollback()
    raise AssertionError("Era esperado um erro de chave duplicada.")

ROLLBACK executado.
Código do erro: 1062
Mensagem: Duplicate entry '999992-999992' for key 'ratings.PRIMARY'


verificar o resultado de rollback

In [228]:
resultado = estado(i)

print(resultado)

assert resultado == {
    "users": 0,
    "movies": 0,
    "ratings": 0
}

connection.rollback()

{'users': 0, 'movies': 0, 'ratings': 0}


### Remoção dos dados de T1

In [229]:
limpar_dados_teste(999991)

Dados de teste 999991 eliminados.


## Concorrência em MySQL

Pretende-se avaliar o comportamento do MySQL perante
múltiplos clientes concorrentes, analisando os tempos
de resposta, os conflitos e a consistência dos dados.

### Configuração dos clientes

In [230]:
config = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": getpass("Password MySQL: "),
    "database": "bda_movielens"
}

NUM_CLIENTES = [1, 5, 10]

### C1 — Leituras concorrentes

Execução simultânea da consulta Q4 por diferentes
clientes, avaliando os tempos de resposta e a
consistência dos resultados.

In [231]:
def cliente_leitura(id_cliente, barreira):

    # Criar uma ligação independente
    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            # Sincronizar os clientes
            barreira.wait()

            # Medir o tempo de resposta
            inicio = perf_counter()

            cursor.execute(q4)
            dados = cursor.fetchall()

            tempo = (perf_counter() - inicio) * 1000

        return {
            "Cliente": id_cliente,
            "Tempo (ms)": tempo,
            "Dados": dados
        }

    finally:
        conn.close()

### Execução dos testes

In [232]:
resultados_c1 = []
resumo_c1 = []

for n in NUM_CLIENTES:

    barreira = Barrier(n, timeout=30)

    # Executar os clientes em simultâneo
    with ThreadPoolExecutor(max_workers=n) as executor:

        resultados = list(executor.map(
            lambda i: cliente_leitura(i, barreira),
            range(1, n + 1)
        ))

    # Verificar a consistência
    consistente = all(
        r["Dados"] == resultados[0]["Dados"]
        for r in resultados
    )

    tempos = [r["Tempo (ms)"] for r in resultados]

    for r in resultados:
        resultados_c1.append({
            "N.º de clientes": n,
            "Cliente": r["Cliente"],
            "Tempo (ms)": round(r["Tempo (ms)"], 2)
        })

    resumo_c1.append({
        "N.º de clientes": n,
        "Tempo médio (ms)": round(sum(tempos) / n, 2),
        "Tempo mínimo (ms)": round(min(tempos), 2),
        "Tempo máximo (ms)": round(max(tempos), 2),
        "Consistência": consistente
    })

    print(f"Teste com {n} cliente(s) concluído.")

Teste com 1 cliente(s) concluído.
Teste com 5 cliente(s) concluído.
Teste com 10 cliente(s) concluído.


### Resultados individuais

In [233]:
df_c1 = pd.DataFrame(resultados_c1)
display(df_c1)

,N.º de clientes,Cliente,Tempo (ms)
0,1,1,89.23
1,5,1,113.56
2,5,2,110.41
3,5,3,121.78
4,5,4,122.58
5,5,5,120.15
6,10,1,165.74
7,10,2,174.13
8,10,3,163.07
9,10,4,173.44


### Resumo dos resultados

In [234]:
df_resumo_c1 = pd.DataFrame(resumo_c1)
display(df_resumo_c1)

,N.º de clientes,Tempo médio (ms),Tempo mínimo (ms),Tempo máximo (ms),Consistência
0,1,89.23,89.23,89.23,True
1,5,117.70,110.41,122.58,True
2,10,168.10,159.20,174.13,True


#### Análise dos resultados

Com o aumento do número de clientes concorrentes,
observou-se um aumento do tempo médio de resposta,
de 89,23 ms para 168,10 ms.

Apesar do aumento do tempo de resposta, todos os
clientes obtiveram resultados consistentes.

Não foram observados erros durante a execução
das consultas concorrentes.

### C1 — Escritas concorrentes

## Encerramento da ligação

In [235]:
connection.close()